--🚕 Projet : Pipeline de Données des Taxis Jaunes de New York

--Ce projet illustre la mise en place d’un pipeline de données complet (Bronze → Silver → Gold) sous Databricks, visant à transformer, nettoyer et analyser les données publiques des taxis jaunes de New York, puis à les exploiter dans un dashboard interactif.

--Le projet est structuré en trois fichiers principaux :

--Bronze Layer : ingestion et stockage des données brutes sans altération.

--Silver Layer : nettoyage et transformation des données selon des règles métier (formats de dates, cohérence des colonnes, traitement des valeurs aberrantes).

--Gold Layer : modélisation analytique avec agrégation et enrichissement des données pour une exploitation optimale par les analystes et décideurs.

In [0]:
-- LA PHASE BRONZE_LAYER
-- Databricks notebook source
-- Créer la base de données puis l'afficher
create database if not exists bronze_db;
show databases;

databaseName
bronze_db
default
gold_db
information_schema
silver_db
tripsdata


In [0]:

-- Spécifier la base de données par défaut
use bronze_db;

SELECT current_database()

current_schema()
bronze_db


In [0]:
%python
# Afficher le contenu du dossier bronze_data
%fs ls /FileStore/tables/nyc_taxi_data/bronze_data/

path,name,size,modificationTime
dbfs:/FileStore/tables/nyc_taxi_data/bronze_data/archive/,archive/,0,1763023214000
dbfs:/FileStore/tables/nyc_taxi_data/bronze_data/taxi_zone_lookup.csv,taxi_zone_lookup.csv,12331,1763022178000


In [0]:
%python

# Supprimer le fichier "yellow_tripdata_2024_02.parquet" dans le dossier bronze_data
#dbutils.fs.rm("dbfs:/FileStore/tables/nyc_taxi_data/bronze_data/yellow_tripdata_2024_01-1.parquet")

In [0]:
%python

# Supprimer le fichier "taxi_zone_lookup.csv" dans le dossier bronze_data
#dbutils.fs.rm("dbfs:/FileStore/tables/nyc_taxi_data/bronze_data/taxi_zone_lookup.csv")

In [0]:
%python

# Affiche la structure des données
#df_fevrier = spark.read.parquet("dbfs:/FileStore/tables/nyc_taxi_data/bronze_data/yellow_tripdata_2024_01.parquet")
#df_fevrier.printSchema()
       

In [0]:
%python
# Script d'ingestion des données
from pyspark.sql.functions import current_timestamp, col, substring_index
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, DoubleType, StringType, TimestampType
import datetime
# 1. Définir explicitement le schema des données
schema = StructType([
     StructField("VendorID", IntegerType(), True),
     StructField("tpep_pickup_datetime", TimestampType(), True),
     StructField("tpep_dropoff_datetime", TimestampType(), True),
     StructField("passenger_count", LongType(), True),
     StructField("trip_distance", DoubleType(), True),
     StructField("RatecodeID", LongType(), True),
     StructField("store_and_fwd_flag", StringType(), True),
     StructField("PULocationID", IntegerType(), True),
     StructField("DOLocationID", IntegerType(), True),
     StructField("payment_type", LongType(), True),
     StructField("fare_amount", DoubleType(), True),
     StructField("extra", DoubleType(), True),
     StructField("mta_tax", DoubleType(), True),
     StructField("tip_amount", DoubleType(), True),
     StructField("tolls_amount", DoubleType(), True),
     StructField("improvement_surcharge", DoubleType(), True),
     StructField("total_amount", DoubleType(), True),
     StructField("congestion_surcharge", DoubleType(), True),
     StructField("Airport_fee", DoubleType(), True)
     ])
 
 # 2. Dossier source
source_folder = "dbfs:/FileStore/tables/nyc_taxi_data/bronze_data/"
filePath = source_folder + "*.parquet"

# 3. Lister les fichiers parquet
files =  [f for f in dbutils.fs.ls(source_folder) if f.name.endswith(".parquet")]
if files:
    print(f"{len(files)} fichiers trouvés, ingestion en cours...")
    # 4. Lire les fichiers avec schéma forcé
    df = spark.read.schema(schema).parquet(filePath)
    # Afficher les 10 premières lignes
    display(df.limit(10))

    # 5. Ajouter une colonne ingestion_timestamp + une colonne nom de fichier
    df = df.withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("file_name", substring_index(col("_metadata.file_path"), "/", -1))

    # -------------------------------
    # 6. Calcul des nombres de lignes
    # -------------------------------

    # Nombre de lignes dans la dataframe pyspark à ingérer
    new_rows_count = df.count()

    # Vérifier si la table existe dans le Catalogue Spark et compter le nombre de lignes existantes dans cette table
    if "raw_trips" in [t.name for t in spark.catalog.listTables("bronze_db")]:
        existing_rows_count = spark.table("bronze_db.raw_trips").count()
    else:
        existing_rows_count = 0

    # Affichage des informations
    print(f"Nombre de lignes à ingérer : {new_rows_count}")
    print(f"Nombre de lignes existantes dans la table : {existing_rows_count}")

    # 7. Sauvegarde des données dans une table Delta
    df.write.format("delta").mode("append").option("mergeSchema", True).saveAsTable("raw_trips")

    # 8. Contrôle de cohérence au niveau des nombres de lignes
    final_rows_count = spark.table("bronze_db.raw_trips").count()
    expected_rows = existing_rows_count + new_rows_count

    if final_rows_count != expected_rows:
        raise Exception(f"Contrôle de cohérence échoué : Le nombre de lignes dans la table finale {final_rows_count} ne correspond pas au nombre de lignes attendues {expected_rows}") 
    else:
        print(f"Contrôle de cohérence réussi : Le nombre de lignes dans la table finale {final_rows_count} correspond au nombre de lignes attendues {expected_rows}")

    # 9. Archiver les fichiers Parquet traités
    archive_folder = source_folder + "archive/run_" + datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S") + "/"

    for f in files:
        dbutils.fs.mv(f.path, archive_folder + f.name)

    print(f"{len(files)} fichiers archivés dans {archive_folder}")

    # Afficher le contenu du dossier archive
    display(dbutils.fs.ls(archive_folder))
else:
    print("Aucun fichier parquet trouvé dans le dossier source donc il n'y a rien à ingérer")

1 fichiers trouvés, ingestion en cours...


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
1,2024-06-01T00:03:46Z,2024-06-01T00:31:23Z,1,12.5,1,N,138,195,1,48.5,7.75,0.5,11.55,0.0,1.0,69.3,0.0,1.75
2,2024-06-01T00:55:22Z,2024-06-01T01:08:24Z,1,4.34,1,N,138,7,1,20.5,6.0,0.5,8.4,0.0,1.0,38.15,0.0,1.75
1,2024-06-01T00:23:53Z,2024-06-01T00:32:35Z,1,1.3,1,N,166,41,1,10.0,1.0,0.5,3.1,0.0,1.0,15.6,0.0,0.0
1,2024-06-01T00:32:24Z,2024-06-01T00:40:06Z,1,1.2,1,N,148,114,1,8.6,3.5,0.5,0.2,0.0,1.0,13.8,2.5,0.0
1,2024-06-01T00:51:38Z,2024-06-01T00:58:17Z,1,1.0,1,N,148,249,1,7.2,3.5,0.5,2.0,0.0,1.0,14.2,2.5,0.0
2,2024-06-01T00:26:13Z,2024-06-01T00:37:21Z,1,1.5,1,N,48,229,1,11.4,1.0,0.5,2.0,0.0,1.0,18.4,2.5,0.0
2,2024-06-01T00:01:04Z,2024-06-01T00:57:48Z,1,18.41,2,N,132,48,1,70.0,0.0,0.5,0.15,6.94,1.0,82.84,2.5,1.75
1,2024-06-01T00:43:55Z,2024-06-01T00:49:03Z,4,1.4,1,N,140,236,1,7.9,3.5,0.5,2.6,0.0,1.0,15.5,2.5,0.0
2,2024-05-31T23:38:07Z,2024-05-31T23:51:42Z,1,2.31,1,N,230,239,1,14.9,1.0,0.5,3.98,0.0,1.0,23.88,2.5,0.0
2,2024-06-01T00:00:09Z,2024-06-01T00:05:11Z,1,0.74,1,N,142,239,1,6.5,1.0,0.5,2.3,0.0,1.0,13.8,2.5,0.0


Nombre de lignes à ingérer : 3539193
Nombre de lignes existantes dans la table : 16792900
Contrôle de cohérence réussi : Le nombre de lignes dans la table finale 20332093 correspond au nombre de lignes attendues 20332093
1 fichiers archivés dans dbfs:/FileStore/tables/nyc_taxi_data/bronze_data/archive/run_2025-11-14-06-27-02/


path,name,size,modificationTime
dbfs:/FileStore/tables/nyc_taxi_data/bronze_data/archive/run_2025-11-14-06-27-02/yellow_tripdata_2024_06.parquet,yellow_tripdata_2024_06.parquet,59859922,1763101623000


In [0]:
-- Afficher les 10 premières lignes de la table raw_trips
SELECT * FROM raw_trips LIMIT 10;

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,ingestion_timestamp,file_name
1,2024-03-01T00:18:51Z,2024-03-01T00:23:45Z,0,1.3,1,N,142,239,1,8.6,3.5,0.5,2.7,0.0,1.0,16.3,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
1,2024-03-01T00:26:00Z,2024-03-01T00:29:06Z,0,1.1,1,N,238,24,1,7.2,3.5,0.5,3.0,0.0,1.0,15.2,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
2,2024-03-01T00:09:22Z,2024-03-01T00:15:24Z,1,0.86,1,N,263,75,2,7.9,1.0,0.5,0.0,0.0,1.0,10.4,0.0,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
2,2024-03-01T00:33:45Z,2024-03-01T00:39:34Z,1,0.82,1,N,164,162,1,7.9,1.0,0.5,1.29,0.0,1.0,14.19,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
1,2024-03-01T00:05:43Z,2024-03-01T00:26:22Z,0,4.9,1,N,263,7,2,25.4,3.5,0.5,0.0,0.0,1.0,30.4,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
2,2024-03-01T00:50:42Z,2024-03-01T01:10:40Z,1,5.04,1,N,238,159,2,25.4,1.0,0.5,0.0,0.0,1.0,27.9,0.0,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
2,2024-03-01T00:08:23Z,2024-03-01T00:17:53Z,1,2.15,1,N,161,141,1,12.1,1.0,0.5,5.13,0.0,1.0,22.23,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
2,2024-03-01T00:24:58Z,2024-03-01T00:30:31Z,1,1.1,1,N,236,237,1,8.6,1.0,0.5,2.04,0.0,1.0,15.64,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
2,2024-03-01T00:49:40Z,2024-03-01T01:01:25Z,1,2.78,1,N,161,114,1,14.9,1.0,0.5,2.0,0.0,1.0,21.9,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet
1,2024-03-01T00:21:43Z,2024-03-01T00:24:44Z,1,0.3,1,N,237,141,2,5.1,3.5,0.5,0.0,0.0,1.0,10.1,2.5,0.0,2025-11-13T10:24:42.105334Z,yellow_tripdata_2024_03.parquet


In [0]:
-- Compter le nombre de ligne dans la table raw_trips
SELECT COUNT(*) FROM raw_trips

count(1)
20332093


In [0]:
-- Afficher les fichiers distincts dans la table raw_trips
SELECT DISTINCT file_name FROM raw_trips ORDER BY file_name;

file_name
yellow_tripdata_2024_01.parquet
yellow_tripdata_2024_02.parquet
yellow_tripdata_2024_03.parquet
yellow_tripdata_2024_04.parquet
yellow_tripdata_2024_05.parquet
yellow_tripdata_2024_06.parquet


In [0]:
-- La sturucture de raw_trips
DESCRIBE EXTENDED raw_trips;

col_name,data_type,comment
VendorID,int,null
tpep_pickup_datetime,timestamp,null
tpep_dropoff_datetime,timestamp,null
passenger_count,bigint,null
trip_distance,double,null
RatecodeID,bigint,null
store_and_fwd_flag,string,null
PULocationID,int,null
DOLocationID,int,null
payment_type,bigint,null


In [0]:
%python
zone_file_path = "dbfs:/FileStore/tables/nyc_taxi_data/bronze_data/taxi_zone_lookup.csv"
df_zones = spark.read.option("header", "true").option("inferSchema", "true").csv(zone_file_path)
# Afficher les 10 premières lignes
display(df_zones.limit(10))


LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


In [0]:
%python
# Sauvegarder les données dans la table taxi_zones
df_zones.write.format("delta").mode("ignore").saveAsTable("taxi_zones")


In [0]:
-- Afficher les 10 premières ligne du fichier taxi_zones
SELECT * FROM taxi_zones LIMIT 10;

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone
